# I. Librerías y configuración inicial

In [ ]:
import hashlib
import pandas as pd
import pyarrow.parquet as pq

## Parámetros y constantes

In [7]:
# ============================================================
# Parámetros generales del análisis
# ============================================================

# Reglas para construir la población de modelamiento
INCLUIR_VALORES_CERO = False
INCLUIR_MULTIPLES_VALORES = False

# Tratamiento de fechas de apertura potencialmente convencionales
TRATAR_FECHAS_APERTURA_SOSPECHOSAS_COMO_NULAS = True

# Partición de los datos
YEAR_OOT = 2023
PROPORCION_TEST = 0.20

# Permite reproducir siempre la misma asignación TRAIN / TEST
SPLIT_SALT = "grupo11_inmobiliario_v1"

# Conversión aproximada de días a años
DIAS_POR_ANIO = 365

## Rutas de acceso

In [ ]:
ruta = "../data/processed/transacciones_bogota_cundinamarca.parquet"

# II. Carga y exploración inicial del archivo

## 2.1 Estructura general de los datos

Antes de definir las tablas de modelación se revisa la estructura general del archivo, incluyendo las variables disponibles y el tamaño de la base. Esto para tener una mirada inicial de qué factores pueden ser útiles para responder el problema planteado

In [8]:
# Revisión de las variables de la tabla de Bogotá y Cundinamarca
archivo = pq.ParquetFile(ruta)
archivo.schema.names

['PK',
 'MATRICULA',
 'FECHA_RADICA_TEXTO',
 'FECHA_APERTURA_TEXTO',
 'YEAR_RADICA',
 'ORIP',
 'DIVIPOLA',
 'DEPARTAMENTO',
 'MUNICIPIO',
 'TIPO_PREDIO_ZONA',
 'CATEGORIA_RURALIDAD',
 'NUM_ANOTACION',
 'ESTADO_FOLIO',
 'FOLIOS_DERIVADOS',
 'Dinámica_Inmobiliaria',
 'COD_NATUJUR',
 'NOMBRE_NATUJUR',
 'NUMERO_CATASTRAL',
 'NUMERO_CATASTRAL_ANTIGUO',
 'DOCUMENTO_JUSTIFICATIVO',
 'COUNT_A',
 'COUNT_DE',
 'PREDIOS_NUEVOS',
 'TIENE_VALOR',
 'TIENE_MAS_DE_UN_VALOR',
 'VALOR']

In [ ]:
print("Número de filas:", archivo.metadata.num_rows)
print("Número de columnas:", archivo.metadata.num_columns)
print("Número de grupos:", archivo.metadata.num_row_groups)

## 2.2 Exploración inicial de la variable objetivo

La reformulación del proyecto plantea un problema de regresión en el que `VALOR`
es la principal candidata a variable respuesta para tener un desarrollo de tipo supervisado. Para una primera revisión se cargan solo las variables asociadas con el valor de la transacción,el tipo de acto jurídico y el año de radicación. Esta extracción parcial es para no tener que trabajar inicialmente con las 26 variables completas cuando todavía no se tiene claridad de los filtros, muestra etc

In [11]:
columnas_objetivo = [
    "VALOR",
    "TIENE_VALOR",
    "TIENE_MAS_DE_UN_VALOR",
    "COD_NATUJUR",
    "NOMBRE_NATUJUR",
    "YEAR_RADICA"
]

df_objetivo = pd.read_parquet(ruta,columns=columnas_objetivo)
display(df_objetivo.head())
df_objetivo.info()

,VALOR,TIENE_VALOR,TIENE_MAS_DE_UN_VALOR,COD_NATUJUR,NOMBRE_NATUJUR,YEAR_RADICA
0,"42,000,000",1,0,125,COMPRAVENTA,2015
1,0,1,0,901,ACLARACIÓN,2015
2,"55,000,000",1,0,125,COMPRAVENTA,2022
3,<NA>,0,0,901,ACLARACIÓN,2018
4,<NA>,0,0,901,ACLARACIÓN,2018


<class 'pandas.DataFrame'>
RangeIndex: 7008966 entries, 0 to 7008965
Data columns (total 6 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   VALOR                  string
 1   TIENE_VALOR            Int64 
 2   TIENE_MAS_DE_UN_VALOR  Int64 
 3   COD_NATUJUR            string
 4   NOMBRE_NATUJUR         string
 5   YEAR_RADICA            Int64 
dtypes: Int64(3), string(3)
memory usage: 598.9 MB


### Configuración de la variable respuesta --> `VALOR`

La variable `VALOR` se encuentra almacenada como tipo texto y contiene separadores de miles en forma de comas. Creamos una versión numérica denominada `VALOR_NUM`, que será utilizada para analizar la calidad de la variable y, luego, como variable respuesta definitiva para el modelamiento

In [13]:
# Comparación columna de y_respuesta original vs la nueva con tipo número
valor_limpio = (
    df_objetivo["VALOR"]
    .str.replace(",", "", regex=False)
    .str.strip()
)
df_objetivo["VALOR_NUM"] = pd.to_numeric(valor_limpio,errors="coerce")
display(df_objetivo[["VALOR", "VALOR_NUM"]].head())

,VALOR,VALOR_NUM
0,"42,000,000",42000000.0
1,0,0.0
2,"55,000,000",55000000.0
3,<NA>,<NA>
4,<NA>,<NA>


### Validación de la conversión

In [15]:
# Se verifica que la transformación no haya generado valores faltantes adicionales
conversion_fallida = (df_objetivo["VALOR"].notna() & df_objetivo["VALOR_NUM"].isna())
print("Valores no nulos que no se pudieron convertir:",conversion_fallida.sum())
print("Total de registros:", len(df_objetivo))
print("VALOR faltante:", df_objetivo["VALOR_NUM"].isna().sum())
print("VALOR = 0:", df_objetivo["VALOR_NUM"].eq(0).sum())
print("VALOR > 0:", df_objetivo["VALOR_NUM"].gt(0).sum())
print("VALOR < 0:", df_objetivo["VALOR_NUM"].lt(0).sum())

Valores no nulos que no se pudieron convertir: 0
Total de registros: 7008966
VALOR faltante: 5112656
VALOR = 0: 806588
VALOR > 0: 1089722
VALOR < 0: 0


## 2.3 Selección del tipo de acto jurídico

Una vez validada la variable `VALOR`, se revisa cómo se distribuyen los valores
faltantes, iguales a cero y positivos entre los diferentes tipos de actos
jurídicos.

El objetivo es identificar un tipo de transacción suficientemente representativo
y con una cantidad adecuada de valores monetarios observados para formular el
problema de regresión.

In [17]:
df_objetivo["ESTADO_VALOR"] = "Faltante"
df_objetivo.loc[df_objetivo["VALOR_NUM"].eq(0),"ESTADO_VALOR"] = "Cero"
df_objetivo.loc[df_objetivo["VALOR_NUM"].gt(0),"ESTADO_VALOR"] = "Positivo"

# Distribución del estado de VALOR por tipo de acto jurídico
tabla_actos = pd.crosstab(df_objetivo["NOMBRE_NATUJUR"],df_objetivo["ESTADO_VALOR"])
display(tabla_actos.sort_values("Positivo", ascending=False).head(20))


ESTADO_VALOR,Cero,Faltante,Positivo
NOMBRE_NATUJUR,,,
COMPRAVENTA,1369,564746,608372
CANCELACIÓN POR VOLUNTAD DE LAS PARTES,49343,708912,182526
COMPRAVENTA DERECHOS DE CUOTA,571,66552,75328
CANCELACIÓN PARCIAL,630,35905,59969
ADJUDICACIÓN EN SUCESION,4021,72278,21088
TRANSFERENCIA DE DOMINIO A TÍTULO DE BENEFICIO EN FIDUCIA MERCANTIL,70,53699,15788
DECLARACIÓN DE CONSTRUCCIÓN EN SUELO PROPIO,420,13401,14013
HIPOTECA ABIERTA SIN LIMITE DE CUANTIA,89688,485987,13207
COMPRAVENTA DERECHOS Y ACCIONES,109,10577,10312


### Selección de compraventas

La comparación muestra que `COMPRAVENTA` es el acto jurídico con la mayor cantidad
de registros que contienen un valor monetario positivo. Por esta razón, se
selecciona este tipo de acto como población base para el problema de regresión.

Esta restricción también permite trabajar con transacciones de naturaleza
comparable, evitando mezclar compraventas con otros actos jurídicos que pueden
tener significados económicos diferentes.

In [18]:
compraventas = df_objetivo[df_objetivo["NOMBRE_NATUJUR"].eq("COMPRAVENTA")].copy()
pd.crosstab(compraventas["ESTADO_VALOR"], compraventas["TIENE_MAS_DE_UN_VALOR"])

TIENE_MAS_DE_UN_VALOR,0,1
ESTADO_VALOR,,
Cero,1369,0
Faltante,564746,0
Positivo,608248,124


### Análisis de compraventas con valor igual a cero

Antes de construir la población de modelamiento se revisan las compraventas cuyo
valor registrado es igual a cero. Estos casos no se eliminan automáticamente,
ya que el valor puede estar asociado a particularidades del proceso registral.

Se revisa su distribución temporal y territorial, así como la posible presencia
de patrones repetidos, para definir posteriormente su tratamiento mediante un
parámetro configurable.

In [19]:
# Se identifican las compraventas cuyo valor registrado es cero
compraventas_cero = compraventas[compraventas["VALOR_NUM"].eq(0)].copy()
print("Total de compraventas con VALOR = 0:",len(compraventas_cero))

display(
    compraventas_cero["YEAR_RADICA"]
    .value_counts()
    .sort_index()
    .rename("Cantidad")
    .to_frame()
)

Total de compraventas con VALOR = 0: 1369


,Cantidad
YEAR_RADICA,
2015,87
2016,110
2017,153
2018,182
2019,146
2020,74
2021,86
2022,476
2023,55


In [20]:
columnas_revision_ceros = [
    "MATRICULA",
    "YEAR_RADICA",
    "ORIP",
    "MUNICIPIO",
    "TIPO_PREDIO_ZONA",
    "NUM_ANOTACION",
    "NOMBRE_NATUJUR",
    "VALOR"
]

ceros_detalle = pd.read_parquet(
    ruta,
    columns=columnas_revision_ceros,
    filters=[
        ("NOMBRE_NATUJUR", "==", "COMPRAVENTA"),
        ("VALOR", "==", "0")
    ]
)

print("Registros recuperados:", len(ceros_detalle))

Registros recuperados: 1369


In [22]:
"""
Revisión de concentración de los casos con 0 en el valor
"""


print("Matrículas únicas:", ceros_detalle["MATRICULA"].nunique())

print("\nPrincipales municipios:")
display(
    ceros_detalle["MUNICIPIO"]
    .value_counts()
    .head(10)
    .to_frame("Cantidad")
)

print("\nPrincipales oficinas de registro:")
display(
    ceros_detalle["ORIP"]
    .value_counts()
    .head(10)
    .to_frame("Cantidad")
)

# Una misma matrícula y anotación puede aparecer más de una vez
duplicados_cero = ceros_detalle.duplicated(subset=["MATRICULA", "NUM_ANOTACION"],keep=False)
print("\nRegistros repetidos por matrícula y anotación:",duplicados_cero.sum())

Matrículas únicas: 1353

Principales municipios:


,Cantidad
MUNICIPIO,
SOACHA,549
LA MESA,319
ANAPOIMA,84
RICAURTE,80
GIRARDOT,65
EL COLEGIO,39
APULO,27
TOCAIMA,26
TENA,20



Principales oficinas de registro:


,Cantidad
ORIP,
51,567
166,527
307,206
170,33
162,18
150,13
167,3
50S,2



Registros repetidos por matrícula y anotación: 18


### Comparación relativa por municipio

La cantidad absoluta de valores cero puede estar influenciada por el volumen de
transacciones de cada municipio. Por esta razón, se compara su frecuencia con el
total de compraventas que tienen un valor monetario conocido.

Esta comparación permite evaluar si la presencia de valores cero se concentra de
forma particular en algunos municipios o si simplemente está asociada a un mayor
número de transacciones.

In [24]:
# Se cargan únicamente las variables necesarias para comparar municipios
comparacion_municipios = pd.read_parquet(
    ruta,
    columns=[
        "MUNICIPIO",
        "NOMBRE_NATUJUR",
        "VALOR"
    ],
    filters=[
        ("NOMBRE_NATUJUR", "==", "COMPRAVENTA")
    ]
)

# Conversión de VALOR a formato numérico
valor_limpio = (
    comparacion_municipios["VALOR"]
    .str.replace(",", "", regex=False)
    .str.strip()
)

comparacion_municipios["VALOR_NUM"] = pd.to_numeric(valor_limpio,errors="coerce")
# Clasificación según disponibilidad del valor
comparacion_municipios["ESTADO_VALOR"] = "Faltante"

comparacion_municipios.loc[comparacion_municipios["VALOR_NUM"].eq(0),"ESTADO_VALOR"] = "Cero"
comparacion_municipios.loc[comparacion_municipios["VALOR_NUM"].gt(0),"ESTADO_VALOR"] = "Positivo"

In [25]:
# Comparación de los estados de VALOR dentro de cada municipio
tabla_municipios = pd.crosstab(comparacion_municipios["MUNICIPIO"],comparacion_municipios["ESTADO_VALOR"])
tabla_municipios["PORCENTAJE_CERO"] = (100* tabla_municipios["Cero"]/ (tabla_municipios["Cero"]+ tabla_municipios["Positivo"]))
display(tabla_municipios.sort_values("Cero", ascending=False).head(20))

ESTADO_VALOR,Cero,Faltante,Positivo,PORCENTAJE_CERO
MUNICIPIO,,,,
SOACHA,549,35,105407,0.518140
LA MESA,319,0,11372,2.728595
ANAPOIMA,84,0,5653,1.464180
RICAURTE,80,15,19712,0.404204
GIRARDOT,65,19,22704,0.285476
EL COLEGIO,39,0,5168,0.748992
APULO,27,0,3133,0.854430
TOCAIMA,26,1,4307,0.600046
TENA,20,0,3143,0.632311


Resultado de la revisión

Los valores cero presentan cierta concentración territorial, aunque representan
una proporción pequeña frente al número de compraventas con valores positivos.
También se identificó concentración en algunas oficinas de registro y un número
reducido de registros repetidos por matrícula y anotación.

Estos patrones sugieren que los valores cero podrían estar asociados a
particularidades en el registro de la información. Sin embargo, con los datos
disponibles no es posible afirmar que correspondan necesariamente a errores.

Por esta razón, su inclusión en la población de modelamiento se mantiene como una
decisión configurable mediante el parámetro `INCLUIR_VALORES_CERO`.

### Definición de las reglas de modelamiento

La revisión de las compraventas identificó dos situaciones que requieren una
decisión antes de construir la población de modelamiento: los registros con valor
igual a cero y las compraventas asociadas a más de un valor monetario.

En las compraventas con valor positivo se identificaron únicamente 124 registros
con más de un valor asociado. Debido a que estos casos pueden representar una
estructura diferente de la variable respuesta, su inclusión se controla mediante
el parámetro `INCLUIR_MULTIPLES_VALORES`.

De forma similar, los valores iguales a cero presentan algunos patrones
territoriales y registrales, pero no existe evidencia suficiente para afirmar que
todos correspondan a errores. Por esta razón, su tratamiento también se mantiene
configurable mediante el parámetro `INCLUIR_VALORES_CERO`.

Con la configuración actual, ambos parámetros se encuentran en `False`, por lo
que la población inicial de modelamiento estará formada por compraventas con un
único valor monetario y con `VALOR_NUM` estrictamente positivo.

# III. Construcción de la población y partición de datos

A partir de las decisiones tomadas en la exploración anterior, se construye la
población de modelamiento utilizando los parámetros definidos al inicio del
notebook.

Posteriormente, los registros entre 2015 y 2022 conforman el periodo de
desarrollo, mientras que 2023 se reserva como conjunto Out-of-Time (OOT) para
evaluar el comportamiento del modelo en un periodo posterior.

## 3.1 Construcción de la población de modelamiento

In [26]:
columnas_particion = [
    "MATRICULA",
    "YEAR_RADICA",
    "NOMBRE_NATUJUR",
    "TIENE_MAS_DE_UN_VALOR",
    "VALOR"
]

df_particion = pd.read_parquet(
    ruta,
    columns=columnas_particion,
    filters=[
        ("NOMBRE_NATUJUR", "==", "COMPRAVENTA")
    ]
)

# VALOR se convierte nuevamente a formato numérico para construir la población
valor_limpio = (
    df_particion["VALOR"]
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_particion["VALOR_NUM"] = pd.to_numeric(
    valor_limpio,
    errors="coerce"
)

print("Compraventas cargadas:", len(df_particion))
print(
    "Matrículas faltantes:",
    df_particion["MATRICULA"].isna().sum()
)

Compraventas cargadas: 1174487
Matrículas faltantes: 0


In [28]:
# Regla para los valores monetarios
if INCLUIR_VALORES_CERO:
    filtro_valor = df_particion["VALOR_NUM"].ge(0)
else:
    filtro_valor = df_particion["VALOR_NUM"].gt(0)

# Regla para los registros asociados a múltiples valores
if INCLUIR_MULTIPLES_VALORES:
    filtro_multiple = pd.Series(
        True,
        index=df_particion.index
    )
else:
    filtro_multiple = (
        df_particion["TIENE_MAS_DE_UN_VALOR"].eq(0)
    )

poblacion_modelamiento = df_particion[
    filtro_valor & filtro_multiple
].copy()

print("Incluir valores cero:", INCLUIR_VALORES_CERO)
print("Incluir múltiples valores:", INCLUIR_MULTIPLES_VALORES)
print("Registros para modelamiento:",len(poblacion_modelamiento))

Incluir valores cero: False
Incluir múltiples valores: False
Registros para modelamiento: 608248


## 3.2 Separación entre desarrollo y OOT

Los registros anteriores a 2023 conforman el periodo de desarrollo. El año 2023
se mantiene completamente separado como conjunto Out-of-Time para evaluar la
capacidad del modelo de generalizar a un periodo posterior.

In [31]:
desarrollo = poblacion_modelamiento[poblacion_modelamiento["YEAR_RADICA"] < YEAR_OOT].copy()
oot = poblacion_modelamiento[poblacion_modelamiento["YEAR_RADICA"].eq(YEAR_OOT)].copy()

print(f"Desarrollo (< {YEAR_OOT}):",len(desarrollo))
print(f"OOT ({YEAR_OOT}):", len(oot))

Desarrollo (< 2023): 450046
OOT (2023): 158202


## 3.3 Partición TRAIN / TEST por matrícula

El periodo de desarrollo se divide entre TRAIN y TEST utilizando la matrícula
inmobiliaria como unidad de asignación. De esta forma, todas las observaciones de
una misma matrícula quedan en un único conjunto y se evita que información del
mismo inmueble aparezca simultáneamente en entrenamiento y prueba.

La asignación se realiza mediante un hash reproducible, utilizando una semilla
textual fija.

In [32]:
def asignar_conjunto(matricula):
    texto = f"{SPLIT_SALT}|{matricula}"

    hash_hex = hashlib.sha256(
        texto.encode("utf-8")
    ).hexdigest()

    numero = int(hash_hex[:8], 16) % 10000
    proporcion = numero / 10000

    if proporcion < PROPORCION_TEST:
        return "TEST"

    return "TRAIN"

In [ ]:
matriculas_desarrollo = (
    desarrollo["MATRICULA"]
    .dropna()
    .drop_duplicates()
)

asignacion_matriculas = {
    matricula: asignar_conjunto(matricula)
    for matricula in matriculas_desarrollo
}

desarrollo["CONJUNTO"] = (desarrollo["MATRICULA"].map(asignacion_matriculas))
train = desarrollo[desarrollo["CONJUNTO"].eq("TRAIN")].copy()
test = desarrollo[desarrollo["CONJUNTO"].eq("TEST")].copy()
oot["CONJUNTO"] = "OOT"

## 3.4 Validación de la partición

Se verifica el tamaño de cada conjunto y el número de matrículas únicas. También
se comprueba que no existan matrículas compartidas entre TRAIN y TEST, ya que la
partición se realizó a nivel de matrícula y no a nivel de fila.

In [35]:
print("TRAIN")
print("Filas:", len(train))
print("Matrículas únicas:", train["MATRICULA"].nunique())

print("\nTEST")
print("Filas:", len(test))
print("Matrículas únicas:", test["MATRICULA"].nunique())

print("\nOOT")
print("Filas:", len(oot))
print("Matrículas únicas:", oot["MATRICULA"].nunique())

matriculas_train = set(train["MATRICULA"].dropna())
matriculas_test = set(test["MATRICULA"].dropna())

interseccion_train_test = matriculas_train & matriculas_test

print(
    "\nMatrículas compartidas entre TRAIN y TEST:",
    len(interseccion_train_test)
)

TRAIN
Filas: 360195
Matrículas únicas: 333195

TEST
Filas: 89851
Matrículas únicas: 83084

OOT
Filas: 158202
Matrículas únicas: 155321

Matrículas compartidas entre TRAIN y TEST: 0


## 3.5 Revisión de matrículas en el conjunto OOT

Aunque el conjunto OOT corresponde completamente al año 2023, algunas matrículas
pueden haber aparecido previamente durante el periodo de desarrollo.

Esta situación no implica por sí sola fuga de información, ya que la separación
del OOT se realiza de forma temporal. Sin embargo, distinguir entre matrículas
con y sin antecedentes permitirá evaluar posteriormente si el modelo tiene un
comportamiento diferente entre ambos grupos.

In [36]:
matriculas_desarrollo_set = set(
    desarrollo["MATRICULA"].dropna()
)

matriculas_oot_set = set(
    oot["MATRICULA"].dropna()
)

matriculas_oot_vistas = (
    matriculas_oot_set
    & matriculas_desarrollo_set
)

matriculas_oot_no_vistas = (
    matriculas_oot_set
    - matriculas_desarrollo_set
)

print(
    "Matrículas OOT vistas previamente en desarrollo:",
    len(matriculas_oot_vistas)
)

print(
    "Matrículas OOT no vistas previamente en desarrollo:",
    len(matriculas_oot_no_vistas)
)

Matrículas OOT vistas previamente en desarrollo: 12775
Matrículas OOT no vistas previamente en desarrollo: 142546


La mayoría de las matrículas presentes en el conjunto OOT no aparecen en el
periodo de desarrollo. Sin embargo, las matrículas que sí tienen antecedentes se
mantienen en OOT, ya que este conjunto se definió mediante una separación
cronológica y no mediante el identificador del inmueble.

Más adelante, el desempeño del modelo podrá evaluarse por separado para
matrículas vistas y no vistas durante el desarrollo.

# IV. Revisión de variables predictoras

En esta sección se revisan las variables disponibles con el fin de definir cuáles
pueden incorporarse a la base de modelación.

La revisión considera su significado, calidad, cardinalidad y posible relación con
la variable objetivo, prestando especial atención a identificadores y variables
que puedan generar fuga de información.

## 4.1 Variables identificadoras

Las variables `PK`, `MATRICULA`, `NUMERO_CATASTRAL` y
`NUMERO_CATASTRAL_ANTIGUO` corresponden principalmente a identificadores de
registros o predios.

Estas variables no se utilizarán directamente como predictores, ya que su alta
cardinalidad podría favorecer la memorización de observaciones particulares en
lugar del aprendizaje de patrones generalizables.

Sin embargo, algunas se conservarán como variables auxiliares. En particular,
`MATRICULA` se utiliza para controlar la separación entre TRAIN y TEST y podría
emplearse posteriormente para construir características históricas del folio.

In [37]:
variables_identificadoras = [
    "PK",
    "MATRICULA",
    "NUMERO_CATASTRAL",
    "NUMERO_CATASTRAL_ANTIGUO"
]

variables_identificadoras

['PK', 'MATRICULA', 'NUMERO_CATASTRAL', 'NUMERO_CATASTRAL_ANTIGUO']

## 4.2 Variables relacionadas con la variable objetivo

Las variables `VALOR`, `VALOR_NUM`, `TIENE_VALOR`,
`TIENE_MAS_DE_UN_VALOR`, `COD_NATUJUR` y `NOMBRE_NATUJUR`
están directamente relacionadas con la construcción de la variable respuesta
o con la definición de la población de modelamiento.

`VALOR_NUM` corresponde a la variable objetivo del problema de regresión y,
por tanto, no puede utilizarse como predictor.

`VALOR` es la fuente original de la variable objetivo, mientras que
`TIENE_VALOR` y `TIENE_MAS_DE_UN_VALOR` se utilizan como información auxiliar
para analizar y construir la población de modelamiento.

Finalmente, `COD_NATUJUR` y `NOMBRE_NATUJUR` no aportan variabilidad una vez
que la población se restringe exclusivamente a actos de tipo `COMPRAVENTA`.

In [38]:
variables_objetivo_auxiliares = [
    "VALOR",
    "VALOR_NUM",
    "TIENE_VALOR",
    "TIENE_MAS_DE_UN_VALOR",
    "COD_NATUJUR",
    "NOMBRE_NATUJUR"
]

variables_objetivo_auxiliares

['VALOR',
 'VALOR_NUM',
 'TIENE_VALOR',
 'TIENE_MAS_DE_UN_VALOR',
 'COD_NATUJUR',
 'NOMBRE_NATUJUR']

## 4.3 Variables temporales

Las variables temporales pueden aportar información sobre cambios en el mercado
inmobiliario y sobre el momento en que ocurre cada transacción.

Se revisan `YEAR_RADICA`, `FECHA_RADICA_TEXTO` y
`FECHA_APERTURA_TEXTO`. Las fechas originales no se utilizarán directamente
como predictores; en su lugar, se construirán variables derivadas que puedan
incorporarse posteriormente a la base de modelación.

### 4.3.1 Preparación de las variables temporales

Para analizar las fechas se carga la misma población definida para modelamiento,
incorporando las variables temporales necesarias. Esto permite que las decisiones
sobre calidad y transformación se realicen sobre los registros que posteriormente
formarán parte de la base de modelación.

In [ ]:
columnas_fechas = [
    "MATRICULA",
    "YEAR_RADICA",
    "FECHA_RADICA_TEXTO",
    "FECHA_APERTURA_TEXTO",
    "NOMBRE_NATUJUR",
    "TIENE_MAS_DE_UN_VALOR",
    "VALOR"
]

# Se cargan las compraventas junto con las variables temporales
df_fechas = pd.read_parquet(
    ruta,
    columns=columnas_fechas,
    filters=[
        ("NOMBRE_NATUJUR", "==", "COMPRAVENTA")
    ]
)

# Se aplican las mismas reglas utilizadas para construir la población de modelamiento
valor_fechas_limpio = (
    df_fechas["VALOR"]
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_fechas["VALOR_NUM"] = pd.to_numeric(valor_fechas_limpio,errors="coerce")

if INCLUIR_VALORES_CERO:
    filtro_valor = df_fechas["VALOR_NUM"].ge(0)
else:
    filtro_valor = df_fechas["VALOR_NUM"].gt(0)

if INCLUIR_MULTIPLES_VALORES:
    filtro_multiple = pd.Series(True,index=df_fechas.index)
else:
    filtro_multiple = (df_fechas["TIENE_MAS_DE_UN_VALOR"].eq(0))

df_fechas = df_fechas[filtro_valor & filtro_multiple].copy()
print("Registros para análisis temporal:", len(df_fechas))

Registros para análisis temporal: 608248


### 4.3.2 Calidad y conversión de las fechas

Las variables de fecha se encuentran almacenadas como texto y presentan diferentes
formatos. Antes de utilizarlas para construir nuevas variables, se revisa su
disponibilidad y se convierten a formato `datetime`.

La fecha de radicación está disponible para la población de modelamiento, mientras
que la fecha de apertura presenta registros faltantes. Estos casos no se eliminan,
ya que la ausencia de la fecha de apertura puede tratarse posteriormente durante
la construcción de las variables temporales.

In [40]:
# Revisión de datos faltantes en las fechas originales
faltantes_fechas = (
    df_fechas[
        [
            "FECHA_RADICA_TEXTO",
            "FECHA_APERTURA_TEXTO"
        ]
    ]
    .isna()
    .sum()
    .to_frame("Cantidad")
)

display(faltantes_fechas)

,Cantidad
FECHA_RADICA_TEXTO,0
FECHA_APERTURA_TEXTO,181764


In [41]:
def convertir_fecha_mixta(serie):
    serie = serie.astype("string").str.strip()

    # Identifica fechas que ya vienen en formato AAAA-MM-DD
    formato_iso = serie.str.match(
        r"^\d{4}-\d{2}-\d{2}",
        na=False
    )

    resultado = pd.Series(
        pd.NaT,
        index=serie.index,
        dtype="datetime64[ns]"
    )

    # Fechas en formato ISO: año-mes-día
    resultado.loc[formato_iso] = pd.to_datetime(
        serie.loc[formato_iso],
        format="mixed",
        dayfirst=False,
        errors="coerce"
    )

    # Otros formatos: día-mes-año
    otros_formatos = ~formato_iso & serie.notna()

    resultado.loc[otros_formatos] = pd.to_datetime(
        serie.loc[otros_formatos],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    return resultado

In [ ]:
df_fechas["FECHA_RADICA"] = convertir_fecha_mixta(df_fechas["FECHA_RADICA_TEXTO"])
df_fechas["FECHA_APERTURA"] = convertir_fecha_mixta(df_fechas["FECHA_APERTURA_TEXTO"])

In [ ]:
# Se verifica que la conversión no haya generado pérdidas adicionales
fallos_radica = (df_fechas["FECHA_RADICA_TEXTO"].notna() & df_fechas["FECHA_RADICA"].isna())
fallos_apertura = (df_fechas["FECHA_APERTURA_TEXTO"].notna() & df_fechas["FECHA_APERTURA"].isna())

print("Fechas de radicación que no se pudieron convertir:",fallos_radica.sum())
print("Fechas de apertura no nulas que no se pudieron convertir:", fallos_apertura.sum())

# Se verifica la consistencia entre YEAR_RADICA y la fecha convertida
comparacion_anio = (
    df_fechas["FECHA_RADICA"].notna()
    & (
        df_fechas["FECHA_RADICA"].dt.year
        != df_fechas["YEAR_RADICA"]
    )
)
print("Registros donde el año de FECHA_RADICA no coincide con YEAR_RADICA:",comparacion_anio.sum())

Fechas de radicación que no se pudieron convertir: 0
Fechas de apertura no nulas que no se pudieron convertir: 0
Registros donde el año de FECHA_RADICA no coincide con YEAR_RADICA: 0


In [44]:
display(
    df_fechas[
        [
            "FECHA_RADICA_TEXTO",
            "FECHA_RADICA"
        ]
    ].head(10)
)

,FECHA_RADICA_TEXTO,FECHA_RADICA
0,01/10/2015,2015-10-01
1,03/11/22,2022-11-03
10,2023-02-10 00:00:00,2023-02-10
11,2023-11-29 00:00:00,2023-11-29
26,2022-04-27 00:00:00,2022-04-27
32,2021-12-22 00:00:00,2021-12-22
33,2023-07-05 00:00:00,2023-07-05
34,2021-12-27 00:00:00,2021-12-27
35,2021-10-06 00:00:00,2021-10-06
37,2021-09-28 00:00:00,2021-09-28


La conversión de las fechas no generó valores faltantes adicionales y el año
obtenido a partir de `FECHA_RADICA` es consistente con `YEAR_RADICA`.Además, se verificó que los registros almacenados originalmente en formato `AAAA-MM-DD` conservaran correctamente el mes y el día. Con las fechas
normalizadas se pueden construir las variables temporales que formarán parte de la base de modelación.

### 4.3.3 Variables derivadas de la fecha de radicación

La fecha de radicación permite representar el momento en que se registra cada
transacción. Se conserva `YEAR_RADICA`, disponible originalmente en la fuente,
y se deriva el mes de radicación a partir de `FECHA_RADICA`.

No se incorpora inicialmente una variable de trimestre, ya que esta puede
obtenerse directamente a partir del mes y aportaría información redundante.

In [45]:
# Mes en el que fue radicada la transacción
df_fechas["MES_RADICA"] = (
    df_fechas["FECHA_RADICA"].dt.month
)

display(
    df_fechas[
        [
            "FECHA_RADICA",
            "YEAR_RADICA",
            "MES_RADICA"
        ]
    ].head(10)
)

,FECHA_RADICA,YEAR_RADICA,MES_RADICA
0,2015-10-01,2015,10
1,2022-11-03,2022,11
10,2023-02-10,2023,2
11,2023-11-29,2023,11
26,2022-04-27,2022,4
32,2021-12-22,2021,12
33,2023-07-05,2023,7
34,2021-12-27,2021,12
35,2021-10-06,2021,10
37,2021-09-28,2021,9


In [46]:
print(
    "Años disponibles:",
    sorted(df_fechas["YEAR_RADICA"].dropna().unique())
)

print(
    "Mes mínimo:",
    df_fechas["MES_RADICA"].min()
)

print(
    "Mes máximo:",
    df_fechas["MES_RADICA"].max()
)

Años disponibles: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Mes mínimo: 1
Mes máximo: 12


Para representar el momento de la transacción se conservarán inicialmente
`YEAR_RADICA` y `MES_RADICA`.

`FECHA_RADICA` se mantiene como variable auxiliar para trazabilidad y para la
construcción de otras características temporales, pero no se utilizará
directamente como predictor.

### 4.3.4 Fecha de apertura y antigüedad del folio

La fecha de apertura permite aproximar cuánto tiempo llevaba abierto el folio
registral al momento de la transacción.

Antes de construir esta variable se revisan dos situaciones: registros donde la
fecha de apertura es posterior a la fecha de radicación y fechas antiguas que
podrían corresponder a valores convencionales de captura.

In [47]:
fechas_apertura_validas = (
    df_fechas["FECHA_APERTURA"].notna()
)

apertura_posterior = (
    fechas_apertura_validas
    & (
        df_fechas["FECHA_APERTURA"]
        > df_fechas["FECHA_RADICA"]
    )
)

print(
    "Registros con fecha de apertura disponible:",
    fechas_apertura_validas.sum()
)

print(
    "Registros donde FECHA_APERTURA > FECHA_RADICA:",
    apertura_posterior.sum()
)

Registros con fecha de apertura disponible: 426484
Registros donde FECHA_APERTURA > FECHA_RADICA: 3309


Se identifican registros en los que la fecha de apertura es posterior a la fecha
de radicación. Estos casos no se eliminan, ya que podrían estar asociados con
particularidades del proceso registral.

Para ellos, la antigüedad del folio se tratará como faltante, evitando generar
valores negativos.

In [48]:
FECHAS_APERTURA_SOSPECHOSAS = pd.to_datetime([
    "1800-01-01",
    "1900-01-01",
    "1901-01-01"
])

df_fechas["FECHA_APERTURA_SOSPECHOSA"] = (
    df_fechas["FECHA_APERTURA"]
    .dt.normalize()
    .isin(FECHAS_APERTURA_SOSPECHOSAS)
    .astype("Int64")
)

print(
    "Fechas de apertura sospechosas:",
    df_fechas["FECHA_APERTURA_SOSPECHOSA"].sum()
)

Fechas de apertura sospechosas: 210


También se identificaron algunas fechas concentradas exactamente en
`1800-01-01`, `1900-01-01` y `1901-01-01`.

Dado que estas fechas podrían representar valores convencionales de captura, se
marcan mediante `FECHA_APERTURA_SOSPECHOSA`. Su tratamiento en el cálculo de la
antigüedad se controla mediante el parámetro
`TRATAR_FECHAS_APERTURA_SOSPECHOSAS_COMO_NULAS`.

* Construcción de variable de antiguedad

In [49]:
# Diferencia entre la radicación y la apertura del folio
df_fechas["ANTIGUEDAD_FOLIO_DIAS"] = (
    df_fechas["FECHA_RADICA"]
    - df_fechas["FECHA_APERTURA"]
).dt.days

# Las diferencias negativas no se consideran antigüedades válidas
df_fechas.loc[
    df_fechas["ANTIGUEDAD_FOLIO_DIAS"] < 0,
    "ANTIGUEDAD_FOLIO_DIAS"
] = pd.NA

# Tratamiento configurable de fechas potencialmente convencionales
if TRATAR_FECHAS_APERTURA_SOSPECHOSAS_COMO_NULAS:
    df_fechas.loc[
        df_fechas["FECHA_APERTURA_SOSPECHOSA"].eq(1),
        "ANTIGUEDAD_FOLIO_DIAS"
    ] = pd.NA

df_fechas["ANTIGUEDAD_FOLIO_ANIOS"] = (
    df_fechas["ANTIGUEDAD_FOLIO_DIAS"]
    / DIAS_POR_ANIO
)

df_fechas["APERTURA_POSTERIOR_RADICA"] = (
    df_fechas["FECHA_APERTURA"]
    > df_fechas["FECHA_RADICA"]
).astype("Int64")

In [50]:
print(
    "Antigüedad disponible:",
    df_fechas["ANTIGUEDAD_FOLIO_ANIOS"].notna().sum()
)

print(
    "Antigüedad faltante:",
    df_fechas["ANTIGUEDAD_FOLIO_ANIOS"].isna().sum()
)

display(
    df_fechas["ANTIGUEDAD_FOLIO_ANIOS"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

Antigüedad disponible: 422965
Antigüedad faltante: 185283


count    422965.000000
mean         13.510927
std          13.938772
min           0.000000
1%            0.194521
5%            0.336986
25%           0.972603
50%           8.416438
75%          25.057534
95%          39.909589
99%          48.236603
max         119.739726
Name: ANTIGUEDAD_FOLIO_ANIOS, dtype: float64

La variable `ANTIGUEDAD_FOLIO_ANIOS` representa el tiempo transcurrido entre la
apertura del folio y la radicación de la transacción. Cuando la fecha de apertura no está disponible, es posterior a la radicación o corresponde a una fecha marcada como potencialmente convencional, la antigüedad
se conserva como valor faltante. Los registros no se eliminan por estas situaciones, ya que las demás variables
pueden seguir aportando información útil para la base de modelación.

## 4.4 Variables geográficas y territoriales

La ubicación de la transacción puede estar asociada con diferencias importantes
en los valores registrados. Por esta razón se revisan las variables territoriales
disponibles antes de definir cuáles se incorporarán a la base de modelación.

Se analizan `ORIP`, `DIVIPOLA`, `DEPARTAMENTO`, `MUNICIPIO`,
`TIPO_PREDIO_ZONA` y `CATEGORIA_RURALIDAD`, considerando principalmente su
disponibilidad, cardinalidad y posible redundancia.

### 4.4.1 Disponibilidad y cardinalidad

In [51]:
columnas_geograficas = [
    "MATRICULA",
    "ORIP",
    "DIVIPOLA",
    "DEPARTAMENTO",
    "MUNICIPIO",
    "TIPO_PREDIO_ZONA",
    "CATEGORIA_RURALIDAD",
    "NOMBRE_NATUJUR",
    "TIENE_MAS_DE_UN_VALOR",
    "VALOR"
]

df_geo = pd.read_parquet(
    ruta,
    columns=columnas_geograficas,
    filters=[
        ("NOMBRE_NATUJUR", "==", "COMPRAVENTA")
    ]
)

In [52]:
valor_geo_limpio = (
    df_geo["VALOR"]
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_geo["VALOR_NUM"] = pd.to_numeric(
    valor_geo_limpio,
    errors="coerce"
)

if INCLUIR_VALORES_CERO:
    filtro_valor = df_geo["VALOR_NUM"].ge(0)
else:
    filtro_valor = df_geo["VALOR_NUM"].gt(0)

if INCLUIR_MULTIPLES_VALORES:
    filtro_multiple = pd.Series(
        True,
        index=df_geo.index
    )
else:
    filtro_multiple = (
        df_geo["TIENE_MAS_DE_UN_VALOR"].eq(0)
    )

df_geo = df_geo[
    filtro_valor & filtro_multiple
].copy()

print(
    "Registros para análisis geográfico:",
    len(df_geo)
)

Registros para análisis geográfico: 608248


In [53]:
variables_geograficas = [
    "ORIP",
    "DIVIPOLA",
    "DEPARTAMENTO",
    "MUNICIPIO",
    "TIPO_PREDIO_ZONA",
    "CATEGORIA_RURALIDAD"
]

resumen_geograficas = pd.DataFrame({
    "Tipo": df_geo[variables_geograficas].dtypes.astype(str),
    "Faltantes": df_geo[variables_geograficas].isna().sum(),
    "Porcentaje_faltante": (
        100
        * df_geo[variables_geograficas].isna().mean()
    ).round(2),
    "Valores_unicos": df_geo[variables_geograficas].nunique()
})

display(resumen_geograficas)

,Tipo,Faltantes,Porcentaje_faltante,Valores_unicos
ORIP,string,0,0.0,17
DIVIPOLA,string,0,0.0,117
DEPARTAMENTO,string,0,0.0,2
MUNICIPIO,string,0,0.0,117
TIPO_PREDIO_ZONA,string,0,0.0,3
CATEGORIA_RURALIDAD,string,0,0.0,4


### 4.4.2 Correspondencia entre variables territoriales

Algunas variables pueden representar diferentes niveles de una misma ubicación.
En particular, `DIVIPOLA`, `MUNICIPIO` y `DEPARTAMENTO` pueden contener
información territorial redundante.

Por esta razón se revisa la correspondencia entre estas variables antes de
seleccionar cuáles serán incluidas en la base de modelación.

In [59]:
# Número de municipios asociados a cada código DIVIPOLA
municipios_por_divipola = (df_geo.groupby("DIVIPOLA")["MUNICIPIO"].nunique())
print("Máximo número de municipios asociados a un DIVIPOLA:",municipios_por_divipola.max())
print("DIVIPOLA asociados a más de un municipio:",(municipios_por_divipola > 1).sum())


# Número de códigos DIVIPOLA asociados a cada municipio
divipola_por_municipio = (df_geo.groupby("MUNICIPIO")["DIVIPOLA"].nunique())
print("\nMáximo número de DIVIPOLA asociados a un municipio:",divipola_por_municipio.max())
print("Municipios asociados a más de un DIVIPOLA:",(divipola_por_municipio > 1).sum())

# Cada municipio debería pertenecer a un único departamento
departamentos_por_municipio = (df_geo.groupby("MUNICIPIO")["DEPARTAMENTO"].nunique())
print("\nMáximo número de departamentos asociados a un municipio:",departamentos_por_municipio.max())
print("Municipios asociados a más de un departamento:",(departamentos_por_municipio > 1).sum())

Máximo número de municipios asociados a un DIVIPOLA: 1
DIVIPOLA asociados a más de un municipio: 0

Máximo número de DIVIPOLA asociados a un municipio: 1
Municipios asociados a más de un DIVIPOLA: 0

Máximo número de departamentos asociados a un municipio: 1
Municipios asociados a más de un departamento: 0


In [60]:
display(
    df_geo[
        [
            "DIVIPOLA",
            "DEPARTAMENTO",
            "MUNICIPIO"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["DEPARTAMENTO", "MUNICIPIO"]
    )
    .head(20)
)

,DIVIPOLA,DEPARTAMENTO,MUNICIPIO
10,11001,BOGOTÁ D.C.,BOGOTÁ D.C.
665091,25001,CUNDINAMARCA,AGUA DE DIOS
667554,25019,CUNDINAMARCA,ALBÁN
665090,25035,CUNDINAMARCA,ANAPOIMA
673986,25040,CUNDINAMARCA,ANOLAIMA
0,25599,CUNDINAMARCA,APULO
677482,25053,CUNDINAMARCA,ARBELÁEZ
679330,25086,CUNDINAMARCA,BELTRÁN
679425,25095,CUNDINAMARCA,BITUIMA
679822,25099,CUNDINAMARCA,BOJACÁ


Los resultados confirman una correspondencia uno a uno entre `DIVIPOLA` y
`MUNICIPIO`: cada código identifica un único municipio y cada municipio está
asociado a un único código DIVIPOLA.

Asimismo, cada municipio pertenece a un único departamento. Por esta razón,
incorporar simultáneamente `DIVIPOLA`, `MUNICIPIO` y `DEPARTAMENTO` como
predictores introduciría información territorial redundante.

Se conservará `MUNICIPIO` como variable candidata para modelamiento por su
interpretabilidad. `DIVIPOLA` se mantendrá como variable auxiliar de identificación
territorial y `DEPARTAMENTO` no se utilizará como predictor directo.

### 4.4.3 Relación entre oficina de registro y municipio

La variable `ORIP` identifica la oficina de registro asociada a la transacción.
Aunque tiene una cardinalidad menor que `MUNICIPIO`, podría aportar información
adicional sobre la organización registral del territorio.

Se revisa su relación con los municipios para determinar si representa información
diferente o si puede derivarse directamente de la ubicación municipal.

In [63]:
# Número de ORIP asociados a cada municipio
orip_por_municipio = (df_geo.groupby("MUNICIPIO")["ORIP"].nunique())
print("Máximo número de ORIP asociados a un municipio:",orip_por_municipio.max())
print("Municipios asociados a más de un ORIP:",(orip_por_municipio > 1).sum())

# Número de municipios atendidos por cada ORIP
municipios_por_orip = (df_geo.groupby("ORIP")["MUNICIPIO"].nunique())
display(
    municipios_por_orip
    .sort_values(ascending=False)
    .to_frame("Municipios")
)

Máximo número de ORIP asociados a un municipio: 4
Municipios asociados a más de un ORIP: 2


,Municipios
ORIP,
156,17
176,10
152,10
172,10
157,9
50N,9
166,8
307,7
162,7


In [66]:
display(orip_por_municipio[orip_por_municipio > 1].sort_values(ascending=False).to_frame("Cantidad_ORIP"))
municipios_multi_orip = (
    orip_por_municipio[
        orip_por_municipio > 1
    ]
    .index
)

display(
    df_geo.loc[
        df_geo["MUNICIPIO"].isin(municipios_multi_orip),
        ["MUNICIPIO", "ORIP"]
    ]
    .drop_duplicates()
    .sort_values(["MUNICIPIO", "ORIP"])
)

,Cantidad_ORIP
MUNICIPIO,
BOGOTÁ D.C.,4
TOCAIMA,2


,MUNICIPIO,ORIP
10,BOGOTÁ D.C.,50C
22124,BOGOTÁ D.C.,50N
26692,BOGOTÁ D.C.,50S
41,BOGOTÁ D.C.,51
1093518,TOCAIMA,150
994356,TOCAIMA,307


Los resultados muestran que `ORIP` y `MUNICIPIO` están relacionados, pero no
existe una correspondencia uno a uno entre ambas variables.

La mayoría de los municipios se asocia a una única oficina de registro, aunque
se identificaron dos municipios atendidos por más de una `ORIP`, alcanzando un
máximo de cuatro oficinas para un mismo municipio. Asimismo, una oficina puede
estar asociada con varios municipios.

Por esta razón, `ORIP` contiene información registral adicional que no puede
obtenerse únicamente a partir de `MUNICIPIO`. Ambas variables se conservarán
como candidatas para la base de modelación.

### 4.4.4 Tipo de predio y categoría de ruralidad

Las variables `TIPO_PREDIO_ZONA` y `CATEGORIA_RURALIDAD` describen
características territoriales asociadas con la ubicación del predio.

Dado que ambas tienen una cardinalidad baja, se revisa su distribución y la
relación entre sus categorías para determinar si aportan información diferente
o si existe una redundancia importante entre ellas.

In [70]:
# Distribución y relación entre las variables de zona y ruralidad

print("TIPO_PREDIO_ZONA")

display(
    df_geo["TIPO_PREDIO_ZONA"]
    .value_counts(dropna=False)
    .to_frame("Cantidad")
)

print("\nCATEGORIA_RURALIDAD")

display(
    df_geo["CATEGORIA_RURALIDAD"]
    .value_counts(dropna=False)
    .to_frame("Cantidad")
)

tabla_zona_ruralidad = pd.crosstab(df_geo["TIPO_PREDIO_ZONA"],df_geo["CATEGORIA_RURALIDAD"])
display(tabla_zona_ruralidad)

TIPO_PREDIO_ZONA


,Cantidad
TIPO_PREDIO_ZONA,
URBANO,508090
RURAL,89206
SIN INFORMACION,10952



CATEGORIA_RURALIDAD


,Cantidad
CATEGORIA_RURALIDAD,
Ciudades y aglomeraciones,487339
Intermedio,94084
Rural,22908
Rural disperso,3917


CATEGORIA_RURALIDAD,Ciudades y aglomeraciones,Intermedio,Rural,Rural disperso
TIPO_PREDIO_ZONA,,,,
RURAL,28830,44300,13177,2899
SIN INFORMACION,9730,859,199,164
URBANO,448779,48925,9532,854


Los resultados muestran que `TIPO_PREDIO_ZONA` y `CATEGORIA_RURALIDAD`
están relacionadas, pero no representan exactamente la misma información.

Cada categoría de `TIPO_PREDIO_ZONA` aparece distribuida entre distintas
categorías de ruralidad. Por ejemplo, los predios clasificados como rurales no
se concentran exclusivamente en las categorías `Rural` o `Rural disperso`,
sino que también aparecen en municipios clasificados como `Intermedio` o
`Ciudades y aglomeraciones`.

Esto sugiere que `TIPO_PREDIO_ZONA` describe una característica asociada al
predio, mientras que `CATEGORIA_RURALIDAD` representa una clasificación más
general del territorio. Por esta razón, ambas variables se conservarán como
candidatas para la base de modelación.